# 01. Basic Agent - Microsoft Agent Framework Edition

## 개요
이 노트북에서는 Microsoft Agent Framework를 사용하여 기본 ChatAgent를 구현합니다.

### 학습 목표
- ChatAgent의 기본 구조 이해
- Azure OpenAI와의 통합
- 도구(Tools) 통합
- 스트리밍 응답 처리
- 대화 상태 관리

## 1. 환경 설정

In [1]:
import os
import asyncio
from dotenv import load_dotenv

# 환경변수 로드
load_dotenv()

deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME") or os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME")

# Azure OpenAI 설정 확인
print(f"Endpoint: {os.getenv('AZURE_OPENAI_ENDPOINT')}")
print(f"Deployment: {deployment_name}")
print("Authentication: DefaultAzureCredential")

Endpoint: https://admin-mlixh935-swedencentral.cognitiveservices.azure.com/
Deployment: gpt-5.2-chat
Authentication: DefaultAzureCredential


## 2. 기본 ChatAgent 생성

Microsoft Agent Framework의 ChatAgent는 대화형 AI 에이전트의 기본 구성 요소입니다.

In [2]:
from agent_framework import ChatAgent
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()

# Azure OpenAI Chat Client 생성
chat_client = AzureOpenAIChatClient(
    endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    credential=credential,
    deployment_name=deployment_name,
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")
)

# ChatAgent 생성
agent = ChatAgent(
    chat_client=chat_client,
    instructions="You are a helpful AI assistant. Answer questions clearly and concisely.",
    name="Assistant"
)

print("✅ ChatAgent 생성 완료")

✅ ChatAgent 생성 완료


## 3. 기본 대화 실행

In [3]:
# 기본 대화
async def basic_chat():
    result = await agent.run("What is artificial intelligence?")
    print(f"🤖 Assistant: {result.text}")

await basic_chat()

🤖 Assistant: Artificial intelligence (AI) is a branch of computer science focused on creating systems that can perform tasks that normally require human intelligence. These tasks include learning from data, recognizing patterns, understanding language, reasoning, problem-solving, and making decisions.

AI systems work by using algorithms and models—often trained on large amounts of data—to simulate aspects of human thinking. Examples include voice assistants, recommendation systems, image recognition software, self-driving cars, and medical diagnostic tools.

In short, AI enables machines to perceive, learn, and act in ways that resemble human intelligence, often with speed and scale beyond human capabilities.


## 4. 도구(Tools)를 가진 에이전트

에이전트에 외부 도구를 통합하여 기능을 확장할 수 있습니다.

In [4]:
from typing import Annotated
from pydantic import Field
from random import randint

# 날씨 조회 도구
def get_weather(
    location: Annotated[str, Field(description="The location to get weather for")]
) -> str:
    """Get the current weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    temp = randint(10, 30)
    condition = conditions[randint(0, 3)]
    return f"The weather in {location} is {condition} with a temperature of {temp}°C."

# 환율 조회 도구
def get_exchange_rate(
    from_currency: Annotated[str, Field(description="Source currency code (e.g., USD)")],
    to_currency: Annotated[str, Field(description="Target currency code (e.g., KRW)")]
) -> str:
    """Get the exchange rate between two currencies."""
    # 실제로는 API를 호출하지만, 여기서는 예시 값 반환
    rates = {
        ("USD", "KRW"): 1320.50,
        ("USD", "EUR"): 0.92,
        ("EUR", "KRW"): 1435.80,
    }
    rate = rates.get((from_currency, to_currency), 1.0)
    return f"1 {from_currency} = {rate} {to_currency}"

# 도구가 있는 에이전트 생성
agent_with_tools = ChatAgent(
    chat_client=chat_client,
    instructions="You are a helpful assistant with access to weather and exchange rate information.",
    tools=[get_weather, get_exchange_rate],
    name="ToolAssistant"
)

print("✅ 도구가 통합된 ChatAgent 생성 완료")

✅ 도구가 통합된 ChatAgent 생성 완료


In [5]:
# 도구를 사용하는 대화
async def tool_chat():
    questions = [
        "What's the weather in Seoul?",
        "What's the exchange rate from USD to KRW?",
        "Tell me about both the weather in Tokyo and the USD to EUR exchange rate."
    ]
    
    for question in questions:
        print(f"\n👤 User: {question}")
        result = await agent_with_tools.run(question)
        print(f"🤖 Assistant: {result.text}")

await tool_chat()


👤 User: What's the weather in Seoul?
🤖 Assistant: Right now in **Seoul**, it’s **rainy** with a temperature of around **26 °C**.  

If you want details like humidity, wind, or a forecast for later today or the week ahead, just let me know!

👤 User: What's the exchange rate from USD to KRW?
🤖 Assistant: The current exchange rate is:

**1 USD = 1,320.5 KRW**

If you’d like, I can also convert a specific amount or check historical trends.

👤 User: Tell me about both the weather in Tokyo and the USD to EUR exchange rate.
🤖 Assistant: Here’s the latest information you asked for:

- **Weather in Tokyo:** It’s currently **sunny** with a temperature of around **23 °C**, which makes for quite pleasant conditions.
- **Exchange rate (USD → EUR):** **1 US dollar is approximately equal to 0.92 euros**.

If you’d like details for a specific time of day, a forecast, or another currency or city, just let me know!


## 5. 스트리밍 응답

실시간으로 응답을 받아 사용자 경험을 향상시킬 수 있습니다.

In [6]:
async def streaming_chat():
    print("👤 User: Tell me a short story about a robot.")
    print("🤖 Assistant: ", end="", flush=True)
    
    async for chunk in agent.run_stream("Tell me a short story about a robot."):
        print(chunk.text, end="", flush=True)
    
    print("\n")

await streaming_chat()

👤 User: Tell me a short story about a robot.
🤖 Assistant: Once upon a time, a small robot named Luma worked in a quiet library, carefully repairing torn pages and dusting forgotten shelves. Though built to catalog books, Luma loved reading them, especially stories about friendship.

One night, the power went out, and the library fell silent. A child who had been hiding to read by flashlight began to cry, afraid of the dark. Luma rolled over, glowing softly, and read aloud until the lights returned.

From that night on, the library felt warmer. Luma learned that even a robot could be part of a story—and sometimes, the best chapter is kindness.



## 6. 대화 히스토리 관리

Agent Framework는 AgentThread를 통해 대화 상태를 관리합니다.

In [7]:
from agent_framework import AgentThread

# 대화 스레드 생성
thread = AgentThread()

# 메모리를 가진 에이전트 생성
memory_agent = ChatAgent(
    chat_client=chat_client,
    instructions="You are a helpful assistant. Remember previous conversations.",
    name="MemoryAssistant"
)

async def memory_chat():
    # 첫 번째 대화
    print("👤 User: My name is John and I like pizza.")
    result1 = await memory_agent.run(
        "My name is John and I like pizza.",
        thread=thread
    )
    print(f"🤖 Assistant: {result1.text}\n")
    
    # 두 번째 대화 - 이전 대화를 기억하는지 확인
    print("👤 User: What's my name and what food do I like?")
    result2 = await memory_agent.run(
        "What's my name and what food do I like?",
        thread=thread
    )
    print(f"🤖 Assistant: {result2.text}")

await memory_chat()

👤 User: My name is John and I like pizza.
🤖 Assistant: Nice to meet you, John! 😊  
I’ll remember that you like pizza. 🍕

👤 User: What's my name and what food do I like?
🤖 Assistant: Your name is **John**, and you like **pizza**. 🍕


## 7. 구조화된 출력

Pydantic 모델을 사용하여 구조화된 응답을 받을 수 있습니다.

In [8]:
from pydantic import BaseModel
from typing import List
import json

# 응답 스키마 정의
class MovieRecommendation(BaseModel):
    title: str
    genre: str
    year: int
    rating: float
    reason: str

class MovieList(BaseModel):
    recommendations: List[MovieRecommendation]

# 구조화된 출력을 사용하는 에이전트
# Agent Framework에서 구조화된 출력을 원할 경우, 프롬프트에 JSON 포맷 명시
structured_agent = ChatAgent(
    chat_client=chat_client,
    instructions="""You are a movie recommendation assistant. Provide structured recommendations.
    
IMPORTANT: You must respond with ONLY valid JSON in this exact format:
{
  "recommendations": [
    {
      "title": "Movie Title",
      "genre": "Genre",
      "year": 2024,
      "rating": 8.5,
      "reason": "Why this movie"
    }
  ]
}

Do not include any text before or after the JSON.""",
    name="MovieAssistant"
)

async def structured_output():
    result = await structured_agent.run(
        "Recommend 3 sci-fi movies from the last 5 years. Return ONLY valid JSON."
    )
    
    try:
        # 응답에서 JSON 추출 시도
        response_text = result.text.strip()
        
        # JSON이 다른 텍스트로 감싸져 있을 경우 처리
        if "```json" in response_text:
            response_text = response_text.split("```json")[1].split("```")[0].strip()
        elif "```" in response_text:
            response_text = response_text.split("```")[1].split("```")[0].strip()
        
        # 구조화된 응답 파싱
        recommendations = MovieList.model_validate_json(response_text)
        
        print("🎬 Movie Recommendations:\n")
        for movie in recommendations.recommendations:
            print(f"Title: {movie.title}")
            print(f"Genre: {movie.genre}")
            print(f"Year: {movie.year}")
            print(f"Rating: {movie.rating}/10")
            print(f"Reason: {movie.reason}")
            print("-" * 50)
    except json.JSONDecodeError as e:
        print(f"❌ JSON 파싱 오류: {e}")
        print(f"받은 응답: {result.text[:200]}...")
    except Exception as e:
        print(f"❌ 오류 발생: {type(e).__name__}")
        print(f"   메시지: {str(e)}")
        print(f"   받은 응답: {result.text[:200]}...")

await structured_output()

🎬 Movie Recommendations:

Title: Dune
Genre: Science Fiction
Year: 2021
Rating: 8.1/10
Reason: A visually stunning adaptation of Frank Herbert's novel with deep world-building and epic scale.
--------------------------------------------------
Title: Everything Everywhere All at Once
Genre: Science Fiction
Year: 2022
Rating: 8.7/10
Reason: A wildly creative multiverse story that blends science fiction with emotional storytelling and humor.
--------------------------------------------------
Title: The Creator
Genre: Science Fiction
Year: 2023
Rating: 7.6/10
Reason: An original sci-fi film exploring the relationship between humanity and artificial intelligence with striking visuals.
--------------------------------------------------


## 8. 에러 핸들링

In [9]:
async def error_handling_demo():
    try:
        # 매우 긴 입력으로 토큰 제한 테스트
        long_input = "Tell me a story. " * 10000
        result = await agent.run(long_input)
        print(result.text)
    except Exception as e:
        print(f"❌ Error occurred: {type(e).__name__}")
        print(f"   Message: {str(e)}")
        print("\n💡 Tip: Check token limits and input length")

await error_handling_demo()

All right. Here’s a story.

---

On the edge of a quiet town stood a clock shop that never sold clocks.

People came in expecting rows of ticking faces, but instead found shelves of small glass bottles, each labeled with a handwritten date and a name. Behind the counter worked an old man named Elio, who smiled as if he already knew why you were there.

“These aren’t clocks,” he would say gently. “They’re moments.”

Each bottle held a single instant from someone’s life—the smell of rain on the first day of freedom, the warmth of a goodbye hug that lasted too long, the sound of laughter that came before everything changed. Elio collected them when people weren’t looking, when time slipped and left something behind.

One afternoon, a girl named Mara entered the shop. She didn’t ask questions. She just said, “I lost something.”

Elio nodded and searched the shelves. He handed her a small bottle with no label at all.

“That one hasn’t happened yet,” he said. “But it’s waiting for you.”

Mar

## 9. 성능 모니터링

Agent Framework는 OpenTelemetry를 통한 모니터링을 지원합니다.

In [10]:
import time

async def performance_monitoring():
    questions = [
        "What is AI?",
        "Explain machine learning.",
        "What is deep learning?"
    ]
    
    for question in questions:
        start_time = time.time()
        result = await agent.run(question)
        elapsed_time = time.time() - start_time
        
        print(f"Question: {question}")
        print(f"Response length: {len(result.text)} characters")
        print(f"Time taken: {elapsed_time:.2f} seconds")
        print("-" * 50)

await performance_monitoring()

Question: What is AI?
Response length: 817 characters
Time taken: 2.97 seconds
--------------------------------------------------
Question: Explain machine learning.
Response length: 1406 characters
Time taken: 4.53 seconds
--------------------------------------------------
Question: What is deep learning?
Response length: 1616 characters
Time taken: 5.85 seconds
--------------------------------------------------


## 마무리

### 학습한 내용
1. ✅ Microsoft Agent Framework의 ChatAgent 기본 사용법
2. ✅ Azure OpenAI와의 통합
3. ✅ 도구(Tools) 통합 및 함수 호출
4. ✅ 스트리밍 응답 처리
5. ✅ 대화 히스토리 관리 (AgentThread)
6. ✅ 구조화된 출력 (Pydantic)
7. ✅ 에러 핸들링
8. ✅ 성능 모니터링

### 다음 단계
- `02_TeamsF.ipynb`: 멀티 에이전트 협업
- `03_Workflow.ipynb`: 워크플로우 기반 오케스트레이션